# Modelo ensemble Ridge–Gradient Boosting para predicción de NDVI

Este notebook implementa el modelo seleccionado después de comparar diferentes variables objetivo, escenarios de variables, algoritmos y combinaciones.

## Configuración seleccionada

- Variable objetivo: NDVI.
- Escenario: clima más historia de NDVI.
- Número de variables predictoras: 39.
- Modelo 1: Ridge, con `alpha = 100`.
- Modelo 2: Gradient Boosting.
- Peso de Ridge: 55 %.
- Peso de Gradient Boosting: 45 %.
- Validación: cinco particiones temporales crecientes.
- Métrica de selección: RMSE de validación.

El conjunto de prueba final se mantiene reservado y no se utiliza en este notebook.

In [1]:
import sys
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, VotingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import make_scorer

# Identificar automáticamente la raíz del repositorio
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    REPO_ROOT = CURRENT_DIR.parent
else:
    REPO_ROOT = CURRENT_DIR

MODELS_PATH = REPO_ROOT / "models"
RESULTS_PATH = REPO_ROOT / "results"
DATASET_PATH = REPO_ROOT / "data" / "processed" / "dataset_modelo.csv"

sys.path.insert(0, str(MODELS_PATH))

from _experiment_utils import prepare_features, split_train_val_test

print("Raíz del repositorio:", REPO_ROOT)
print("Dataset encontrado:", DATASET_PATH.exists())
print("Carpeta models encontrada:", MODELS_PATH.exists())

Raíz del repositorio: c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe
Dataset encontrado: True
Carpeta models encontrada: True


In [3]:
df = pd.read_csv(
    DATASET_PATH,
    parse_dates=["window_start", "window_end"]
)

train_val, test_final = split_train_val_test(
    df,
    fecha_col="window_start",
    frac_test_final=0.15
)

print("Dimensiones del dataset:", df.shape)

print(
    "Periodo de entrenamiento y validación:",
    train_val["window_start"].min(),
    "a",
    train_val["window_start"].max()
)

print(
    "Periodo reservado para prueba final:",
    test_final["window_start"].min(),
    "a",
    test_final["window_start"].max()
)

print("Filas de entrenamiento y validación:", len(train_val))
print("Filas reservadas para prueba final:", len(test_final))

split_train_val_test (fecha_col='window_start', frac_test_final=0.15):
  train_val:  2000-03-21 00:00:00 -> 2022-07-28 00:00:00  (1030 filas)
  test_final: 2022-08-13 00:00:00 -> 2026-07-12 00:00:00  (182 filas)
Dimensiones del dataset: (1212, 48)
Periodo de entrenamiento y validación: 2000-03-21 00:00:00 a 2022-07-28 00:00:00
Periodo reservado para prueba final: 2022-08-13 00:00:00 a 2026-07-12 00:00:00
Filas de entrenamiento y validación: 1030
Filas reservadas para prueba final: 182


## Preparación del escenario seleccionado

`prepare_features` excluye el NDVI actual, el EVI actual, las fechas y los identificadores que no deben utilizarse como predictores.

A partir del conjunto completo se excluyen las variables contemporáneas e históricas de LAI y EVI. Se conservan las variables climáticas, estacionales, regionales y la historia previa de NDVI.

In [4]:
X_completo, y_ndvi = prepare_features(
    train_val,
    target_col="ndvi"
)

# Variables que no pertenecen al escenario seleccionado
columnas_excluir = [
    "lai_high_lag0",
    "lai_high_lag1",
    "lai_high_lag2",
    "evi_lag_1year",
    "evi_lag1w"
]

columnas_excluir = [
    columna
    for columna in columnas_excluir
    if columna in X_completo.columns
]

X_ndvi = X_completo.drop(
    columns=columnas_excluir
).reset_index(drop=True)

y_ndvi = y_ndvi.reset_index(drop=True)

print("Objetivo: NDVI")
print("Número de observaciones:", len(y_ndvi))
print("Número de variables:", X_ndvi.shape[1])
print("Variables excluidas:", columnas_excluir)

assert X_ndvi.shape[1] == 39, (
    f"Se esperaban 39 variables, pero se encontraron {X_ndvi.shape[1]}"
)

print("\nValores faltantes por variable:")
display(
    X_ndvi.isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

Objetivo: NDVI
Número de observaciones: 1030
Número de variables: 39
Variables excluidas: ['lai_high_lag0', 'lai_high_lag1', 'lai_high_lag2', 'evi_lag_1year', 'evi_lag1w']

Valores faltantes por variable:


deficit_hidrico_trend2y    44
ndvi_lag_1year             42
et_mm_lag1                  2
dewpoint_c_lag0             0
precip_mm_lag0              0
pet_mm_lag0                 0
tmean_c_lag0                0
soil_moist_layer3_lag0      0
soil_moist_layer2_lag0      0
soil_moist_layer1_lag0      0
dtype: int64

In [7]:
fechas_train_val = pd.to_datetime(
    train_val["window_start"]
).reset_index(drop=True)

fechas_unicas = np.sort(
    fechas_train_val.unique()
)

test_size_fechas = len(fechas_unicas) // 6

divisor_temporal = TimeSeriesSplit(
    n_splits=5,
    test_size=test_size_fechas
)

cv_temporal = []

for numero_fold, (
    idx_fechas_train,
    idx_fechas_val
) in enumerate(divisor_temporal.split(fechas_unicas), start=1):

    fechas_fold_train = fechas_unicas[idx_fechas_train]
    fechas_fold_val = fechas_unicas[idx_fechas_val]

    idx_train = np.flatnonzero(
        fechas_train_val.isin(fechas_fold_train).to_numpy()
    )

    idx_val = np.flatnonzero(
        fechas_train_val.isin(fechas_fold_val).to_numpy()
    )

    cv_temporal.append((idx_train, idx_val))

    fecha_inicio_train = pd.Timestamp(
        fechas_fold_train.min()
    ).date()

    fecha_fin_train = pd.Timestamp(
        fechas_fold_train.max()
    ).date()

    fecha_inicio_val = pd.Timestamp(
        fechas_fold_val.min()
    ).date()

    fecha_fin_val = pd.Timestamp(
        fechas_fold_val.max()
    ).date()

    print(
        f"Fold {numero_fold}: "
        f"train={len(idx_train)} filas, "
        f"validación={len(idx_val)} filas | "
        f"{fecha_inicio_train} a {fecha_fin_train} -> "
        f"{fecha_inicio_val} a {fecha_fin_val}"
    )

Fold 1: train=180 filas, validación=170 filas | 2000-03-21 a 2004-02-02 -> 2004-02-18 a 2007-10-16
Fold 2: train=350 filas, validación=170 filas | 2000-03-21 a 2007-10-16 -> 2007-11-01 a 2011-06-26
Fold 3: train=520 filas, validación=170 filas | 2000-03-21 a 2011-06-26 -> 2011-07-12 a 2015-03-06
Fold 4: train=690 filas, validación=170 filas | 2000-03-21 a 2015-03-06 -> 2015-03-22 a 2018-11-17
Fold 5: train=860 filas, validación=170 filas | 2000-03-21 a 2018-11-17 -> 2018-12-03 a 2022-07-28


## Definición del modelo seleccionado

El ensamble combina:

1. Ridge, que presentó el mejor desempeño individual y la menor brecha entre entrenamiento y validación.
2. Gradient Boosting, que aportó predicciones complementarias y permitió mejorar el desempeño del ensamble.

Los parámetros y pesos permanecen fijos. En este notebook no se realiza una nueva búsqueda de hiperparámetros.

In [8]:
RANDOM_STATE = 42

modelo_ridge = Pipeline([
    (
        "imputador",
        SimpleImputer(strategy="median")
    ),
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        Ridge(alpha=100.0)
    )
])

modelo_gradient_boosting = Pipeline([
    (
        "imputador",
        SimpleImputer(strategy="median")
    ),
    (
        "modelo",
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=10,
            subsample=0.85,
            random_state=RANDOM_STATE
        )
    )
])

modelo_ensemble = VotingRegressor(
    estimators=[
        ("ridge", modelo_ridge),
        ("gradient_boosting", modelo_gradient_boosting)
    ],
    weights=[0.55, 0.45],
    n_jobs=1
)

print("Modelo definido correctamente.")
print("Peso Ridge: 0.55")
print("Peso Gradient Boosting: 0.45")

Modelo definido correctamente.
Peso Ridge: 0.55
Peso Gradient Boosting: 0.45


In [9]:
def pearson_seguro(y_true, y_pred):
    """Calcula Pearson evitando errores por series constantes."""

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0

    return float(
        np.corrcoef(y_true, y_pred)[0, 1]
    )


metricas = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "pearson": make_scorer(pearson_seguro)
}

resultados_cv = cross_validate(
    estimator=modelo_ensemble,
    X=X_ndvi,
    y=y_ndvi,
    cv=cv_temporal,
    scoring=metricas,
    return_train_score=True,
    n_jobs=-1
)

tabla_folds = pd.DataFrame({
    "fold": range(1, 6),
    "rmse_train": -resultados_cv["train_rmse"],
    "rmse_validacion": -resultados_cv["test_rmse"],
    "mae_validacion": -resultados_cv["test_mae"],
    "r2_validacion": resultados_cv["test_r2"],
    "pearson_validacion": resultados_cv["test_pearson"]
})

display(tabla_folds.round(4))

,fold,rmse_train,rmse_validacion,mae_validacion,r2_validacion,pearson_validacion
0,1,0.0166,0.0215,0.0169,0.6748,0.8448
1,2,0.0181,0.0204,0.0159,0.5268,0.7338
2,3,0.0183,0.0197,0.0152,0.7156,0.8501
3,4,0.0185,0.0265,0.0191,0.5063,0.7342
4,5,0.0200,0.0191,0.0151,0.5581,0.7842


In [10]:
rmse_train = -resultados_cv["train_rmse"].mean()
rmse_cv = -resultados_cv["test_rmse"].mean()
rmse_std = resultados_cv["test_rmse"].std()
mae_cv = -resultados_cv["test_mae"].mean()
r2_cv = resultados_cv["test_r2"].mean()
pearson_cv = resultados_cv["test_pearson"].mean()

rmse_pct_media = 100 * rmse_cv / y_ndvi.mean()
riesgo_base_proxy = 1 - pearson_cv**2
calidad_datos = X_ndvi.notna().all(axis=1).mean()
brecha_rmse = rmse_cv - rmse_train

cumple_r2 = r2_cv >= 0.60
cumple_rmse = rmse_pct_media <= 15
cumple_pearson = pearson_cv >= 0.65
cumple_riesgo = riesgo_base_proxy <= 0.50
cumple_calidad = calidad_datos >= 0.90

tabla_resultado = pd.DataFrame([{
    "modelo": "Ensemble Ridge 55% + Gradient Boosting 45%",
    "objetivo": "NDVI",
    "escenario": "clima_mas_historia_ndvi",
    "variables": X_ndvi.shape[1],
    "rmse_train": rmse_train,
    "rmse_cv": rmse_cv,
    "rmse_std": rmse_std,
    "rmse_pct_media": rmse_pct_media,
    "mae_cv": mae_cv,
    "r2_cv": r2_cv,
    "pearson_cv": pearson_cv,
    "riesgo_base_proxy": riesgo_base_proxy,
    "calidad_datos": calidad_datos,
    "brecha_rmse": brecha_rmse,
    "cumple_r2": cumple_r2,
    "cumple_rmse": cumple_rmse,
    "cumple_pearson": cumple_pearson,
    "cumple_riesgo_base": cumple_riesgo,
    "cumple_calidad": cumple_calidad,
    "cumple_todas": (
        cumple_r2
        and cumple_rmse
        and cumple_pearson
        and cumple_riesgo
        and cumple_calidad
    )
}])

display(tabla_resultado.round(4))

,modelo,objetivo,escenario,variables,rmse_train,rmse_cv,rmse_std,rmse_pct_media,mae_cv,r2_cv,pearson_cv,riesgo_base_proxy,calidad_datos,brecha_rmse,cumple_r2,cumple_rmse,cumple_pearson,cumple_riesgo_base,cumple_calidad,cumple_todas
0,Ensemble Ridge 55% + Gradient Boosting 45%,NDVI,clima_mas_historia_ndvi,39,0.0183,0.0215,0.0027,2.8464,0.0164,0.5963,0.7894,0.3768,0.9573,0.0032,False,True,True,True,True,False


In [11]:
rmse_train = -resultados_cv["train_rmse"].mean()
rmse_cv = -resultados_cv["test_rmse"].mean()
rmse_std = resultados_cv["test_rmse"].std()
mae_cv = -resultados_cv["test_mae"].mean()
r2_cv = resultados_cv["test_r2"].mean()
pearson_cv = resultados_cv["test_pearson"].mean()

rmse_pct_media = 100 * rmse_cv / y_ndvi.mean()
riesgo_base_proxy = 1 - pearson_cv**2
calidad_datos = X_ndvi.notna().all(axis=1).mean()
brecha_rmse = rmse_cv - rmse_train

cumple_r2 = r2_cv >= 0.60
cumple_rmse = rmse_pct_media <= 15
cumple_pearson = pearson_cv >= 0.65
cumple_riesgo = riesgo_base_proxy <= 0.50
cumple_calidad = calidad_datos >= 0.90

tabla_resultado = pd.DataFrame([{
    "modelo": "Ensemble Ridge 55% + Gradient Boosting 45%",
    "objetivo": "NDVI",
    "escenario": "clima_mas_historia_ndvi",
    "variables": X_ndvi.shape[1],
    "rmse_train": rmse_train,
    "rmse_cv": rmse_cv,
    "rmse_std": rmse_std,
    "rmse_pct_media": rmse_pct_media,
    "mae_cv": mae_cv,
    "r2_cv": r2_cv,
    "pearson_cv": pearson_cv,
    "riesgo_base_proxy": riesgo_base_proxy,
    "calidad_datos": calidad_datos,
    "brecha_rmse": brecha_rmse,
    "cumple_r2": cumple_r2,
    "cumple_rmse": cumple_rmse,
    "cumple_pearson": cumple_pearson,
    "cumple_riesgo_base": cumple_riesgo,
    "cumple_calidad": cumple_calidad,
    "cumple_todas": (
        cumple_r2
        and cumple_rmse
        and cumple_pearson
        and cumple_riesgo
        and cumple_calidad
    )
}])

display(tabla_resultado.round(4))

,modelo,objetivo,escenario,variables,rmse_train,rmse_cv,rmse_std,rmse_pct_media,mae_cv,r2_cv,pearson_cv,riesgo_base_proxy,calidad_datos,brecha_rmse,cumple_r2,cumple_rmse,cumple_pearson,cumple_riesgo_base,cumple_calidad,cumple_todas
0,Ensemble Ridge 55% + Gradient Boosting 45%,NDVI,clima_mas_historia_ndvi,39,0.0183,0.0215,0.0027,2.8464,0.0164,0.5963,0.7894,0.3768,0.9573,0.0032,False,True,True,True,True,False


In [12]:
# Entrenar con todo el periodo de entrenamiento y validación
modelo_final = clone(modelo_ensemble)

modelo_final.fit(
    X_ndvi,
    y_ndvi
)

artefacto_modelo = {
    "modelo": modelo_final,
    "variables": X_ndvi.columns.tolist(),
    "objetivo": "ndvi",
    "escenario": "clima_mas_historia_ndvi",
    "numero_variables": X_ndvi.shape[1],

    "parametros_ridge": {
        "alpha": 100.0
    },

    "parametros_gradient_boosting": {
        "n_estimators": 100,
        "learning_rate": 0.03,
        "max_depth": 2,
        "min_samples_leaf": 10,
        "subsample": 0.85,
        "random_state": 42
    },

    "pesos": {
        "ridge": 0.55,
        "gradient_boosting": 0.45
    },

    "metricas_cv": tabla_resultado.iloc[0].to_dict(),

    "periodo_entrenamiento": {
        "inicio": str(train_val["window_start"].min()),
        "fin": str(train_val["window_start"].max())
    }
}

MODELS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

ruta_modelo = (
    MODELS_PATH
    / "ensemble_ridge_gradient_boosting_ndvi.joblib"
)

ruta_metricas = (
    RESULTS_PATH
    / "metricas_ensemble_ndvi_cv.csv"
)

ruta_folds = (
    RESULTS_PATH
    / "metricas_ensemble_ndvi_por_fold.csv"
)

joblib.dump(
    artefacto_modelo,
    ruta_modelo
)

tabla_resultado.to_csv(
    ruta_metricas,
    index=False
)

tabla_folds.to_csv(
    ruta_folds,
    index=False
)

print("Modelo guardado en:")
print(ruta_modelo)

print("\nMétricas guardadas en:")
print(ruta_metricas)

print("\nResultados por fold guardados en:")
print(ruta_folds)

Modelo guardado en:
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\models\ensemble_ridge_gradient_boosting_ndvi.joblib

Métricas guardadas en:
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\results\metricas_ensemble_ndvi_cv.csv

Resultados por fold guardados en:
c:\Users\danie\OneDrive\Documentos\Maestría_MIAD\4 Semestre\Proyecto aplicado en analítica de datos\agroindice-cafe\results\metricas_ensemble_ndvi_por_fold.csv


## Conclusión preliminar

El ensamble ponderado de Ridge y Gradient Boosting fue seleccionado por presentar el mejor desempeño durante la validación temporal.

El modelo cumplió los criterios de RMSE, correlación de Pearson, proxy del componente de diseño del riesgo base y calidad de los datos. El R² quedó muy cerca de la meta establecida, pero no la alcanzó formalmente.

Este notebook documenta la configuración seleccionada y entrena el artefacto definitivo sin utilizar el conjunto de prueba final. La evaluación de dicho periodo se realizará separadamente y no se empleará para reajustar el modelo.